# धडा 18: क्रिप्टोग्राफिक रिसीट्ससह AI एजंट्सचे संरक्षण

## प्रायोगिक नोटबुक

ही नोटबुक चार व्यायामांमधून मार्गदर्शन करते:

1. एजंट टूल कॉलसाठी आपले पहिले रिसीट **स्वाक्षरी करा** आणि त्याची पडताळणी करा.
2. रिसीटमध्ये **तपासणी करा** आणि पडताळणी अयशस्वी होताना पहा.
3. तीन रिसीट्सची साखळी तयार करा आणि साखळीची अखंडता **पुष्टी करा**.
4. Microsoft एजंट फ्रेमवर्क टूल कॉलला वॉराप करा जेणेकरून प्रत्येक क्रियेमुळे एक रिसीट तयार होईल.

सर्व क्रिप्टोग्राफिक प्राथमिकता चांगल्या देखभाल होणाऱ्या लायब्रेर्‍यांमधून आयात केल्या आहेत (`pynacl` Ed25519 साठी, `jcs` RFC 8785 कॅनॉनिकल JSON साठी, `hashlib` Python स्टँडर्ड लायब्ररीमधून SHA-256 साठी). रिसीट लॉजिक स्वतः सामान्य Python आहे ज्याला आपण वाचू आणि सुधारू शकता.

कोशिंबा क्रमाने चालवा. प्रत्येक विभाग छोटा आणि स्वयंपूर्ण आहे.


## सेटअप

दोन अवलंबन स्थापित करा. दोन्ही परवान्याना मोकळा परवाना आहे (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## सहाय्यक उपयुक्तता

हे दोन सहाय्यक बेस64url एनकोडिंग (पॅडिंगशिवाय) आणि मनमानी वस्तूंच्या SHA-256 हॅशिंगची हाताळणी करतात. ते नोटबुकच्या उर्वरित भागाला फक्त रसीद लॉजिकवर लक्ष केंद्रित ठेवतात.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: आपला पहिला पावतीवर स्वाक्षरी करा

कल्पना करा की **Contoso Travel** साठी आमचा एजन्टने सिडनीहून लॉस एंजेलिसकडे प्रवासासाठी उडाण शोधली आहे. आम्हाला या टूल कॉलचे साइन केलेले पावती म्हणून नोंद करायची आहे जेणेकरून भविष्यातील ऑडिटर त्याची पुष्टी आमच्यावर विश्वास न ठेवता करू शकेल.

### Step 1.1: स्वाक्षरीसाठी की तयार करा

उत्पादनात, एजन्टची स्वाक्षरी की हार्डवेअर सिक्युरिटी मॉड्यूल (HSM), Azure Key Vault, किंवा त्यासारख्या संरक्षित स्टोअरमध्ये राहील. या धड्यात आम्ही मेमरीमध्ये नवीन की तयार करतो.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: रिसीट पेलोड तयार करा

पेलोडमध्ये ते सर्व काही असते ज्याची आम्हाला रिसीटद्वारे पुष्टी करायची असते: कोणाने क्रिया केली, कोणते टूल वापरले, कोणते आर्ग्युमेंट्स वापरले, काय परत आले, कोणत्या धोरणाखाली, आणि कधी. आम्ही आर्ग्युमेंट्स आणि निकालाचे हॅश करतो, त्यांना थेट समाविष्ट न करता, जेणेकरून रिसीट संवेदनशील माहिती लीक होणार नाही.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: पावतीवर स्वाक्षरी करा आणि एकत्र करा

तीन टप्पे:

1. JCS वापरून पेलोड कॅनॉनिकलाइझ करा जेणेकरून दोन वेगवेगळ्या अंमलबजावणी एकाच तार्किक पावतीसाठी एकसारखे बाइट्स तयार करतील.
2. SHA-256 सह कॅनॉनिकल बाइट्सचे हॅश करा.
3. Ed25519 खाजगी कळीसह हॅशवर स्वाक्षरी करा.

स्वाक्षरी नंतर मूळ पेलोडशी जोडली जाते ज्यामुळे अंतिम पावती तयार होते.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: पावतीची पडताळणी करा

पडताळणी प्रक्रियेला उलट करते. आम्ही स्वाक्षरी काढून टाकतो, प्रमाणात्मक हॅश पुन्हा गणना करतो, आणि पावतीतील सार्वजनिक कीसह स्वाक्षरी तपासतो.

ही पडताळणी करणारा लेखापरीक्षक आमच्याकडून फक्त पावतीची गरज असते. कोणतीही सेवा कॉल करण्याची गरज नाही, की संचिका तपासण्याची गरज नाही, कोणतीही विश्वास ठेवण्याची गरज नाही.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

आपण पाहिले पाहिजे `Receipt is valid: True`. एजंटने आपला पहिला क्रिप्टोग्राफिकली साईन केलेला ऑडिट रेकॉर्ड तयार केला आहे.


## Section 2: पावतीसह छेडछाड करा

पावत्या यांचे पूर्ण उद्दिष्ट म्हणजे त्या छेडछाड-पुरावा असतील. चला ते सिद्ध करूया.

आम्ही पावतीतील नक्की एक अक्षर बदलू आणि पडताळणी अपयशी होणार ते पाहू.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### काय नुकतेच घडले?

जेव्हा आपण `policy_id` बदलले, तेव्हा कॅनॉनिकल बाइट्स बदलली. त्या बाइट्सचा SHA-256 हॅश बदलला. स्वाक्षरी (जी मूळ हॅशवर होती) नवीन हॅशशी आता जुळत नाही. पडताळणी बरोबरच `False` परत करते.

रिकीटचा कोणताही क्षेत्र बदलेल आणि तरीही ते पडताळणी करेल यासाठी कोणताही मार्ग नाही, जोपर्यंत हल्लेखोराजवळ खासगी की नाही. जोपर्यंत खासगी की एका की वॉल्टमध्ये आहे आणि सार्वजनिक की प्रकाशित केली आहे, तोपर्यंत छेडछाड लपवणे शक्य नाही.

स्वतः प्रयत्न करा: वरच्या सेलमध्ये `tool_name` किंवा `agent_id` किंवा `timestamp` बदला आणि पुन्हा चालवा. प्रत्येक बदल अमान्य रसीद निर्माण करतो.


## विभाग 3: रसीद एकत्र करा

एकल रसीद एक क्रिया संरक्षित करते. बहुतेक एजंट अनेक क्रिया करतात. संपूर्ण अनुक्रम ताम्हेर-प्रमाणित करण्यासाठी, आम्ही प्रत्येक रसीदेमध्ये मागील रसीदचा हॅश नवीन रसीदच्या पेलोडमध्ये समाविष्ट करून मागील रसीदशी लिंक करतो.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

जर कोणी रसीद काढून टाकली किंवा पुन्हा क्रम लावली, तर साखळी अगदी त्या बिंदूवर तुटते. नंतरच्या कोणत्याही रसीदेसाठी पडताळणी अयशस्वी होते कारण तिचा `previous_receipt_hash` मूळ मागील रसीदच्या हॅशशी जुळत नाही.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

आता तुम्ही मधल्या पावतीमध्ये फेरफार करून चेन तोडा आणि पुन्हा पडताळणी करा. फेरफार केलेली पावती तिच्या स्वाक्षऱ्याच्या तपासणीत अपयशी ठरते, आणि पुढील पावती तिच्या चेन-लिंक तपासणीत अपयशी ठरते (कारण तिचा `previous_receipt_hash` आता बदललेल्या मधल्या पावतीच्या हॅशशी जुळत नाही).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

रसीद 0 अद्याप सत्यापित होते (ती बदललेली नाही आणि त्यावर अवलंबून असलेली कोणतीही पूर्वीची रसीद नाही). रसीद 1 च्या सहीची तपासणी अयशस्वी होते कारण आपण `tool_args_hash` बदलले आहे. रसीद 2 च्या साखळी-लिंक तपासणी अयशस्वी होते कारण त्याचा `previous_receipt_hash` मूळ (आता बदललेल्या) रसीद 1 विरुद्ध गणना केला गेला होता.

भलेही हल्लेखोरने बदललेल्या रसीद 1 ला पुन्हा सही केली (जी ते खाजगी कीशिवाय करू शकत नाहीत), तरीही रसीद 2 मधील साखळी-लिंक विसंगती फेरफार उघड करेल. बदल लपवण्यासाठी, हल्लेखोरला बदलाच्या बिंदूपासून सर्व रसीद पुन्हा सही कराव्या लागतील, ज्यासाठी त्यांना खाजगी कीची मालकी लागत आहे.


## विभाग 4: एजंट टूल कॉलसह रिसीट साइनिंग लपेटा

खऱ्या डिप्लॉयमेंटमध्ये, प्रत्येक एजंट लेखकाला `make_receipt` कॉल करण्याची आठवण ठेवायची नसते. आपण प्रत्येक टूल कॉलसाठी रिसीट साइनिंग आपोआप व्हावे अशी इच्छा ठेवतो.

येथे सर्वात सोपा नमुना आहे: एक रॅपर क्लास जो कोणत्याही कॉल करण्यायोग्य टूल फंक्शनला घेऊन त्याची रिसीट-निर्गमित आवृत्ती परत करतो. हे Microsoft Agent Framework (`agent_framework.azure`) सह कोणत्याही एजंट फ्रेमवर्कशी जुळते.

जर तुमच्याकडे Azure AI Foundry प्रोजेक्ट सेटअप नसेल, तरी खालील स्थानिक मॉक हा नमुना दाखवतो.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework सह एकत्रीकरण

वरील `ReceiptedTool` रॅपर फ्रेमवर्क-स्वतंत्र आहे. Microsoft Agent Framework सह बनवलेल्या एजंटमध्ये ते वापरण्यासाठी, रॅप केलेल्या फंक्शनला टूल म्हणून नोंदणी करा. एक आराखडा (आपण मॉकचा वास्तविक Azure AI Foundry टूल नोंदणीने बदलाल):

```python
# एकत्रीकरण आकार दर्शवित pseudocode.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="तुम्ही एक Contoso ट्रॅव्हल एजंट आहात ...",
#     tools=[receipted_lookup],   # वळलेले साधन, मूळ फंक्शन नाही
# )
# response = agent.run("सिडनीहून लॉस एंजेलिसला जूनमध्ये फ्लाइट शोधा.")
#
# # रननंतर, एजंटने केलेल्या प्रत्येक साधन कॉलला सही केलेला पावती असते:
# audit_chain = receipted_lookup.receipts
```

एजंट फ्रेमवर्कला पावत्या (receipts) विषयी काहीही माहिती असण्याची गरज नाही. पावती स्वाक्षरी टूलच्या भोवती रॅप केलेली आहे, फ्रेमवर्कमध्ये जुळवलेली नाही. एजंट पुन्हा लिहिण्याशिवाय विद्यमान एजंट कोडमध्ये हा प्रकार जोडण्याचा मार्ग आहे.


## पुनरावलोकन आणि विस्तार आव्हान

तुमच्याकडे आहे:

- Ed25519 की जोड्या तयार केली.
- एजंट टूल कॉलसाठी रिसीप्ट तयार केला आणि सही केला.
- फक्त सार्वजनिक की वापरून रिसीप्ट ऑफलाइन पडताळणी केली.
- रिसीप्ट मध्ये फेरफार केला आणि पडताळणी अयशस्वी झाली पाहिली.
- तीन रिसीप्ट्सची हॅश-चेन साखळी तयार केली.
- साखळीच्या मध्यभागी फेरफार केला आणि दोन्ही सही अयशस्वी व साखळी लिंक अयशस्वी पाहिला.
- टूल फंक्शनवर स्वयंचलित रिसीप्ट सही करण्याचे रॅप केले.

**विस्तार आव्हान.** रिसीप्ट स्कीमात `request_id` फील्ड (वितरित ट्रेसिंगसाठी UUID) जोडा. `make_receipt` मध्ये ते समाविष्ट करा आणि रिसीप्ट्स अखेरपर्यंत पडताळणी होतात का याची खात्री करा. नंतर सही केल्यानंतर फील्ड बदलून पडताळणी अयशस्वी होते याची खात्री करा. हे तुम्हाला canonical encoding मधील प्रत्येक बाइट कसा सहीसाठी योगदान देतो हे समजायला मदत करेल.

**महत्त्वाचा मर्यादा.** रिसीप्ट तीन गोष्टी सिद्ध करतात आणि फक्त त्या तीनच गोष्टी: attribution (ही की या सामग्रीवर सही केली आहे), integrity (सामग्री सही झाल्यापासून बदललेली नाही), आणि ordering (हा रिसीप्ट त्या रिसीप्टनंतर आला आहे). ते सिद्ध करत नाहीत की एजंटची क्रिया बरोबर होती, `policy_id` मध्ये नमूद केलेली धोरण प्रत्यक्षात अमलात आणली गेली, किंवा एजंटने प्रत्येक नियम पाळला. रिसीप्ट हे पाया आहेत. आढावा घेणे हा प्रणाली आहे ज्यावर तुम्ही बांधता.

त्या मर्यादेशी लक्षात घेऊन धडा README पुन्हा वाचा. टीम्ससाठी रिसीप्टसंबंधी सर्वात सामान्य चूक म्हणजे "आपल्याकडे रिसीप्ट्स आहेत" याचा अर्थ "आपल्यावर नियंत्रण आहे" असा समज होतो. तसे नाही. रिसीप्ट एजंटच्या वर्तनाची ऑडिट करण्यायोग्यता बनवतात. ते ते बरोबर आहेत असे नाही बनवतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
